# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [3]:
from pandas import DataFrame

team_setup = {
    'team_name': '67',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Aidan Liu', 'github': 'Arti-Dev'},
        {'name': 'Siwon Choi', 'github': 'C-1Blue'},
    ],
    'repo_url': 'https://github.com/Arti-Dev/walmart-hurricane-project',                           #
    'charter_url': 'https://github.com/Arti-Dev/walmart-hurricane-project/blob/main/CHARTER.md',
    'notebook_owner_this_week': 'Aidan',           # nice to have
    'meeting_time': 'TODO',                       # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': ['TODO'],              # nice to have, five specific problems
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  not yet - meeting time
  not yet - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [4]:
from pathlib import Path
import subprocess

# import subprocess
#
# REPO_PATH = '../walmart-hurricane-project'      # your local clone of the team repo
#
# result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
#                         capture_output=True, text=True)
# print(result.stdout.strip() or 'no commits found for CHARTER.md')
#
# the above code does not work for our repository.

# the below code was generated by github copilot to prove that we have modified the charter file

result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"Could not find the repo root: {result.stderr.strip()}")

REPO_PATH = Path(result.stdout.strip())
result = subprocess.run(
    ["git", "-C", str(REPO_PATH), "log", "--oneline", "--", "CHARTER.md"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"Could not check CHARTER.md commits: {result.stderr.strip()}")

print(result.stdout.strip() or "no commits found for CHARTER.md")

1a4a836 fix formatting on charter
421ed5c finish charter
503bdd2 Setup


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [29]:
DECISIONS = []
def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [30]:
import pandas as pd
import numpy as np

def remove_duplicates(df):
    df = df.copy()
    after = df.drop_duplicates()
    removed = len(df) - len(after)
    log('duplicates', 'dropped exact duplicate rows', removed)
    return after


def clean_unit_price(df, column ='unit_price'):
    before = df.copy()
    df = df.copy()
    df[column] = df[column].astype(str).str.replace('$', '', regex=False).str.strip()
    df[column] = pd.to_numeric(df[column], errors='coerce')
    log('clean unit prices', 'strip and to numeric', len(df.compare(before)))
    return df

def neg_quant(df, column = 'quantity'):
    before = df.copy()
    df = df.copy()
    if column in df.columns:
        df = df[df[column]>=0]
    removed = len(before) - len(df)
    log('negative quantity', 'drop negative quantities', removed)
    return df

def clean_store_id(df, column = 'store_id'):
    before = df.copy()
    df = df.copy()
    if column in df.columns:
        df[column] = df[column] = df[column].astype(str).str.strip().str.upper()
    log('clean store id', 'strip and upper', len(df.compare(before)))
    return df

def parse_timestamp(df, column = 'timestamp'):
    before = df.copy()
    df = df.copy()
    if column in df.columns:
        df = df.copy()
        if column in df.columns:
                    df[column] = pd.to_datetime(df[column], errors='coerce')
    log('parse timestamps', 'converts to datetime', len(df.compare(before)))
    return df




### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [36]:
# TODO: print or display your decision log
DECISIONS = []

path = '../data/walmart_transactions.csv'
df = pd.read_csv(path)

df = remove_duplicates(df)
df = clean_unit_price(df)
df = neg_quant(df)
df = clean_store_id(df)
df = parse_timestamp(df)

def show_log():
    return pd.DataFrame(DECISIONS)

print(df.shape)

show_log()

[duplicates] dropped exact duplicate rows (30 row(s))
[clean unit prices] strip and to numeric (25532 row(s))
[negative quantity] drop negative quantities (130 row(s))
[clean store id] strip and upper (1785 row(s))
[parse timestamps] converts to datetime (22026 row(s))
(25402, 8)


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,clean unit prices,strip and to numeric,25532
2,negative quantity,drop negative quantities,130
3,clean store id,strip and upper,1785
4,parse timestamps,converts to datetime,22026


### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [8]:
def create_mappings(from_list: list[str], to: str) -> dict[str, str]:
    d = {}
    for fr in from_list:
        d[fr] = to
    return d

CANON = {}
# |= merges dictionaries
CANON |= create_mappings(["SKU#459812", "POPTART-STRAW", "PT_STRAWBERRY"],
                         "PT-12")
CANON |= create_mappings(["WTR-24", "WATER24PK", "WTR_24PK"],
                         "WTR-24")
CANON |= create_mappings(['BG-MONO', 'MONOPOLY-01', 'GAME-MONO'],
                         'BG-MONO')
CANON

{'SKU#459812': 'PT-12',
 'POPTART-STRAW': 'PT-12',
 'PT_STRAWBERRY': 'PT-12',
 'WTR-24': 'WTR-24',
 'WATER24PK': 'WTR-24',
 'WTR_24PK': 'WTR-24',
 'BG-MONO': 'BG-MONO',
 'MONOPOLY-01': 'BG-MONO',
 'GAME-MONO': 'BG-MONO'}

In [38]:
def consolidate(frame: DataFrame):
    out = frame.copy()
    out['canonical_sku'] = out.apply(apply_canonical_sku, axis=1)
    return out

def apply_canonical_sku(row):
    if row['sku'] in CANON:
        return CANON[row['sku']]
    return row['sku']

consolidated = consolidate(df)
consolidated

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,canonical_sku
0,TXN012636,FL-710,2024-09-24 09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15,PNUTBTR-16
1,TXN000759,FL-412,NaT,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02,CNSOUP-01
2,TXN003338,FL-105,NaT,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47,TOILETPAPER12
3,TXN025020,FL-330,NaT,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59,ROKU-STD
4,TXN015797,FL-302,NaT,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82,BEER-24
...,...,...,...,...,...,...,...,...,...
25557,TXN020201,FL-105,2024-09-03 18:28:07,DIAPERS-3,Diapers Size 3,baby,2,24.32,DIAPERS-3
25558,TXN014835,FL-239,NaT,CARDS-01,Playing Cards,Games & Entertainment,3,5.00,CARDS-01
25559,TXN002434,FL-239,NaT,PAPERTOWEL6,Paper Towels 6-Roll,household,2,7.78,PAPERTOWEL6
25560,TXN014739,FL-710,NaT,CHRG-PORT,PORTABLE CHARGER,electronics,1,24.18,CHRG-PORT


In [39]:
consolidated['canonical_sku'].unique()

<StringArray>
[   'PNUTBTR-16',     'CNSOUP-01', 'TOILETPAPER12',      'ROKU-STD',
       'BEER-24',   'CHIPVARIETY',       'TARP-HD',   'PAPERTOWEL6',
   'CANDLE-EMRG',      'CARDS-01',      'CHIPS-VP',     'HDTARP-01',
    'WHITEBREAD',   'PORTCHARGER',      'PTOWEL-6',  'FIRSTAID-KIT',
         'PT-12',      'LTBEER24',      'FLASH-01',         'PB-16',
   'BATT_AA_4PK',      'BFORM-01',        'WTR-24',     'STREAM-01',
      'BATT-100',       'CNDL-EM',         'TP-12',   'BABYFORMULA',
       'DIAP-S3',     'DIAPERS-3',      'SOUP-CAN',  'PLAYCARD-STD',
     'CHRG-PORT',       'BG-MONO',       'BAT-AA4',     'LED-FLASH',
       'FAID-01',      'BREAD-WH']
Length: 38, dtype: str

### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [10]:
import time
import requests

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}
FALLBACK = pd.DataFrame({
    'date': pd.date_range('2024-09-01', periods=30),
    'precipitation_sum': [3.0, 12.4, 24.9, 22.1, 23.1, 9.3, 6.8, 94.7, 28.0, 14.4,
                          12.0, 0.7, 0.6, 21.1, 1.2, 7.9, 2.3, 13.3, 1.4, 0.7,
                          0.4, 2.2, 1.3, 0.3, 6.4, 15.2, 2.5, 18.0, 4.6, 3.3],
    'windspeed_10m_max': [18.8, 13.5, 22.8, 15.5, 15.2, 9.0, 14.2, 12.6, 11.5, 14.3,
                          15.8, 12.1, 10.4, 17.4, 9.0, 13.3, 12.4, 9.3, 12.9, 15.3,
                          19.9, 17.9, 16.6, 17.7, 27.2, 48.8, 45.1, 19.8, 11.7, 8.6],
    'source': 'orlando fallback',
})

def daily_weather(start, end, lat, lon, tries=3):
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'precipitation_sum,windspeed_10m_max',
        'timezone': 'America/New_York',
    }
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
        frame = pd.DataFrame(r.json()['daily'])
        frame = frame.rename(columns={'time': 'date'})
        frame['date'] = pd.to_datetime(frame['date'])
        frame['source'] = 'open-meteo'
        return frame

In [11]:
try:
    weather_orlando = daily_weather('2024-09-01', '2024-09-30', lat=28.54, lon=-81.38)
except requests.RequestException as err:
    print('API call failed:', err)
    weather_orlando = FALLBACK

weather_orlando

,date,precipitation_sum,windspeed_10m_max,source
0,2024-09-01,3.0,18.8,open-meteo
1,2024-09-02,12.4,13.5,open-meteo
2,2024-09-03,24.8,22.8,open-meteo
3,2024-09-04,22.1,15.5,open-meteo
4,2024-09-05,22.7,15.2,open-meteo
5,2024-09-06,9.8,9.0,open-meteo
6,2024-09-07,6.8,14.2,open-meteo
7,2024-09-08,93.7,12.6,open-meteo
8,2024-09-09,29.0,11.5,open-meteo
9,2024-09-10,14.4,14.3,open-meteo


In [12]:
transactions = pd.read_csv("../data/walmart_transactions.csv")

transactions['timestamp'] = pd.to_datetime(transactions['timestamp'], format='mixed')

transactions['date'] = transactions['timestamp'].dt.normalize()
transactions

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,date
0,TXN012636,FL-710,2024-09-24 09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15,2024-09-24
1,TXN000759,fl-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02,2024-09-13
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47,2024-09-17
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59,2024-09-11
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82,2024-09-26
...,...,...,...,...,...,...,...,...,...
25557,TXN020201,FL-105,2024-09-03 18:28:07,DIAPERS-3,Diapers Size 3,baby,2,24.32,2024-09-03
25558,TXN014835,FL-239,2024-09-26 08:47:08,CARDS-01,Playing Cards,Games & Entertainment,3,5.00,2024-09-26
25559,TXN002434,fl-239,2024-09-16 16:39:00,PAPERTOWEL6,Paper Towels 6-Roll,household,2,7.78,2024-09-16
25560,TXN014739,FL-710,2024-09-25 17:08:35,CHRG-PORT,PORTABLE CHARGER,electronics,1,24.18,2024-09-25


Since our data is only in Florida and the open-meteo requests are already set to America/New York timezone, timezones should be fine. I also parsed the transaction data and converted them to timestamps/dates.

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [13]:
# todo Have not started yet

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [40]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': 25402,        # an integer
    'decisions_logged': 5,           # how many entries in your log
    'canonical_products': 38,         # distinct products after consolidation
    'weather_api_rows': 30,           # rows returned by your API call
    'notebook_runs_clean': True,       # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

REQUIRED
  PASS - team_setup_complete = True
  PASS - charter_committed = True
PROGRESS (fill in what you have)
  done - rows_after_cleaning = 25402
  done - decisions_logged = 5
  done - canonical_products = 38
  done - weather_api_rows = 30
  done - notebook_runs_clean = True

required items complete: True
progress items done: 5 of 5


True

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** _..._

**Plan, with an owner and a date:** _..._

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | not started / in progress / draft answer | _..._ |
| Q2 SKU consolidation | _..._ | _..._ |
| Q3 weather correlation | _..._ | _..._ |
| Q4 store and geography | _..._ | _..._ |
| Q5 signal or artifact | _..._ | _..._ |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.